# CIFAKE · ResNet18 fine-tuning and model-behaviour investigation

This is a separate experiment from `baseline_training.ipynb`. It uses the **same audited split**: 88,702 training images, 10,000 validation images, and the untouched official 20,000-image test set. A pretrained ResNet18 learns a single REAL logit. The best epoch is selected by validation loss with a fixed probability threshold of 0.5. The comparison changes the architecture, initialization and input preprocessing together; it is not a controlled architecture-only experiment.

**To review the completed model:** leave `MODE = 'review'` and run the cells. The notebook downloads the checksummed trained weights and saved analysis from the [team model release](https://github.com/adamp-22/kandy/releases/tag/resnet18-plain-seed1-v1) when they are available. The downloadable team ZIP includes the notebook, weights and analysis for offline use. Review mode does not fine-tune a network or download the CIFAKE dataset.

**To run or resume locally:** the prepared data must be in `data/baseline_seed1`. Set `MODE` to `train` or `resume` and `START_BACKGROUND = True`; run the setup/implementation cells and the launch cell once. `finalize` reruns the evaluation/export stage after training has completed, using saved predictions when available. A process lock prevents two runs from training into the same directory.

## 1. Environment

The run was prepared with PyTorch 2.15.0 and torchvision 0.29.1 on an Apple GPU. CPU review is supported. The optional installer uses a compatible PyTorch range because wheel availability differs by operating system. Choose the notebook's Python environment before running it; all implementation code is contained here.

In [1]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import sys,subprocess,importlib.util
    if importlib.util.find_spec('pip') is None:
        subprocess.check_call([sys.executable,'-m','ensurepip','--upgrade'])
    subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.14,<2.16','torchvision==0.29.1',
                           'numpy>=2.0','pillow>=11','scikit-learn>=1.5','matplotlib>=3.9','ipywidgets>=8',
                           'nbformat>=5','nbclient>=0.10','ipykernel>=6'])

## 2. Configuration and training strategy

We start from torchvision's [ImageNet ResNet18 weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html). Two epochs train only the new classifier, with the backbone and BatchNorm running statistics frozen. The remaining epochs fine-tune all layers, including BatchNorm, using a smaller learning rate for the backbone. AdamW, a validation-loss scheduler, gradient clipping and early stopping provide a conservative starting configuration. There is no random flipping, cropping, colour jitter or other training augmentation.

Every native 32×32 RGB image is resized directly to 224×224, then normalized with the ImageNet mean and standard deviation. This intentionally preserves the full frame instead of using the pretrained weights' default resize-and-centre-crop recipe. Upsampling adds no new image detail. All robustness edits and occlusion masks are applied at native 32×32 **before** this fixed preprocessing.

The default cap is 25 epochs and six hours of training time, with early stopping after five full-fine-tuning epochs without sufficient validation-loss improvement. A time limit can stop at a batch checkpoint and select the best fully validated epoch. `PUBLISH_TO_GITHUB` controls the finished model-package upload; the GitHub CLI must already be authenticated for publishing.

In [2]:
from pathlib import Path
from io import BytesIO,StringIO
from datetime import datetime,timezone
from contextlib import contextmanager
import os,sys,time,json,csv,hashlib,random,shutil,subprocess,signal,fcntl,copy,base64,zipfile,urllib.request,urllib.error
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from PIL import Image,ImageFilter,features as pillow_features
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,roc_auc_score
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
else:
    plt.switch_backend('Agg')

ROOT=Path.cwd().resolve()
NOTEBOOK_PATH=ROOT/'resnet18_finetuning.ipynb'
RUN_NAME='resnet18_plain_seed1'
RUN_DIR=ROOT/'runs'/RUN_NAME
DATA_DIR=ROOT/'data'/'baseline_seed1'
SHARE_DIR=RUN_DIR/'share'
MODEL_SAVES=ROOT/'model_saves'
WEIGHTS_PATH=MODEL_SAVES/f'{RUN_NAME}.pt'
MODE='review'                    # review, train, resume, or finalize
DOWNLOAD_SHARED_MODEL=True       # Download the completed, checksummed model package in review mode.
START_BACKGROUND=False          # Set True to run train/resume/finalize independently of the notebook kernel.
PUBLISH_TO_GITHUB=True           # The overnight pipeline uploads the finished model package to the team release.
REPOSITORY='adamp-22/kandy'
RELEASE_TAG='resnet18-plain-seed1-v1'
DEVICE='auto'
MANIFEST_SHA256='098210d89f9441376008d520f72aeef5f65a7aa6b5e0073ba4e875c979e1f205'
TRAINING=dict(seed=1,epochs=25,warmup_epochs=2,batch_size=64,backbone_lr=1e-4,head_lr=1e-3,
              weight_decay=1e-4,patience=5,min_delta=1e-4,checkpoint_batches=200,max_hours=6.0)
torch.set_num_threads(4)

In [3]:
def utc_now():return datetime.now(timezone.utc).isoformat()
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def atomic_json(path,value):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_name(p.name+'.tmp');temp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');temp.replace(p)
def atomic_torch(path,value):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);temp=p.with_name(p.name+'.tmp')
    with temp.open('wb') as f:torch.save(value,f);f.flush();os.fsync(f.fileno())
    temp.replace(p)
def device_for(name='auto'):
    if name=='auto':name='cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    return torch.device(name)
def pfake(z):return np.exp(-np.logaddexp(0,np.asarray(z,dtype=np.float64)))
def table(headers,rows):
    display(Markdown('\n'.join(['| '+' | '.join(headers)+' |','| '+' | '.join(['---']*len(headers))+' |']+
                              ['| '+' | '.join(map(str,row))+' |' for row in rows])))
def model_digest(net):
    h=hashlib.sha256()
    for key,value in sorted(net.state_dict().items()):
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

class CifakeResNet18(nn.Module):
    """Native RGB [0,1] -> full-frame 224 resize -> ImageNet normalization -> one REAL logit."""
    def __init__(self,pretrained=False):
        super().__init__()
        if pretrained:torch.hub.set_dir(str(ROOT/'runs'/'_pretrained_cache'))
        self.net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.net.fc=nn.Linear(self.net.fc.in_features,1)
        self.register_buffer('mean',torch.tensor([.485,.456,.406])[None,:,None,None])
        self.register_buffer('std',torch.tensor([.229,.224,.225])[None,:,None,None])
    def preprocess(self,x):
        return (F.interpolate(x,size=(224,224),mode='bilinear',align_corners=False,antialias=True)-self.mean)/self.std
    def forward(self,x):return self.net(self.preprocess(x)).squeeze(1)
    def forward_maps(self,x):
        b=self.net;x=self.preprocess(x);stem=b.relu(b.bn1(b.conv1(x)))
        last=b.layer4(b.layer3(b.layer2(b.layer1(b.maxpool(stem)))))
        pooled=b.avgpool(last).flatten(1)
        return stem,last,pooled,b.fc(pooled).squeeze(1)
    def continue_stem(self,stem):
        b=self.net;last=b.layer4(b.layer3(b.layer2(b.layer1(b.maxpool(stem)))))
        return b.fc(b.avgpool(last).flatten(1)).squeeze(1)

def load_split(split,data_dir=DATA_DIR,expected=MANIFEST_SHA256):
    directory=Path(data_dir);audit=json.loads((directory/'audit.json').read_text())
    if sha256(directory/'manifest.csv')!=expected:raise ValueError('The data split is different from the baseline split.')
    for filename in [f'{split}_images.npy',f'{split}_labels.npy']:
        if sha256(directory/filename)!=audit['file_sha256'][filename]:raise ValueError(f'Data integrity failure: {filename}')
    images=np.load(directory/f'{split}_images.npy',allow_pickle=False)
    labels=np.load(directory/f'{split}_labels.npy',allow_pickle=False)
    with (directory/'manifest.csv').open(newline='') as f:rows=[r for r in csv.DictReader(f) if r['split']==split]
    assert images.shape==(len(labels),3,32,32) and images.dtype==np.uint8
    assert np.array_equal(labels,[int(r['label']) for r in rows])
    return images,labels,rows

def metrics(labels,z):
    labels=np.asarray(labels,dtype=int);z=np.asarray(z,dtype=np.float64);pred=(z>=0).astype(int)
    assert np.isfinite(z).all() and len(z)==len(labels)
    cm=confusion_matrix(labels,pred,labels=[0,1])
    return dict(n=len(labels),loss=float(np.mean(np.logaddexp(0,z)-labels*z)),accuracy=float(np.mean(pred==labels)),
                balanced_accuracy=float(np.mean(np.diag(cm)/cm.sum(1))),f1_fake=float(f1_score(labels,pred,pos_label=0)),
                roc_auc_fake=float(roc_auc_score(labels==0,-z)),fake_detection_rate=float(cm[0,0]/cm[0].sum()),
                real_flagged_fake_rate=float(cm[1,0]/cm[1].sum()),brier_score_fake=float(np.mean((pfake(z)-(labels==0))**2)),
                confusion_matrix=cm.tolist(),confusion_matrix_order=['FAKE','REAL'])

@torch.inference_mode()
def predict(net,images,selected_device,batch_size=64):
    net.eval();result=[]
    for start in range(0,len(images),batch_size):
        x=torch.tensor(np.ascontiguousarray(images[start:start+batch_size]),device=selected_device,dtype=torch.float32)/255
        result.append(net(x).cpu().numpy())
    return np.concatenate(result)

def rng_state(device):
    return {'python':random.getstate(),'torch':torch.get_rng_state(),
            'accelerator':torch.mps.get_rng_state() if device.type=='mps' else torch.cuda.get_rng_state() if device.type=='cuda' else None}
def restore_rng(state,device):
    random.setstate(state['python']);torch.set_rng_state(state['torch'].cpu())
    if state['accelerator'] is not None:
        if device.type=='mps':torch.mps.set_rng_state(state['accelerator'].cpu())
        elif device.type=='cuda':torch.cuda.set_rng_state(state['accelerator'].cpu())
def configure_phase(net,epoch,cfg):
    warmup=epoch<=cfg['warmup_epochs']
    for name,p in net.named_parameters():p.requires_grad_(not warmup or name.startswith('net.fc.'))
    net.eval() if warmup else net.train()
    net.net.fc.train()
    return 'head_warmup' if warmup else 'full_finetune'

def train_resnet(run_dir=RUN_DIR,data_dir=DATA_DIR,cfg=None,resume=False,selected_device=None,
                 pretrained=True,max_steps_for_check=None):
    """Atomic, resumable updates including permutation, next batch and optimizer/RNG state."""
    cfg=dict(TRAINING if cfg is None else cfg);run_dir=Path(run_dir);data_dir=Path(data_dir)
    device=device_for(DEVICE) if selected_device is None else torch.device(selected_device)
    expected=MANIFEST_SHA256
    if (run_dir/'training_complete.json').exists():return json.loads((run_dir/'training_complete.json').read_text())
    if resume:
        saved=torch.load(run_dir/'last.pt',map_location='cpu',weights_only=True)
        if saved['training']!=cfg or saved['manifest_sha256']!=expected:raise ValueError('Resume configuration or split differs.')
    else:
        if (run_dir/'last.pt').exists() or (run_dir/'config.json').exists():raise ValueError('Run exists; use resume or a new RUN_NAME.')
        saved=None
    random.seed(cfg['seed']);np.random.seed(cfg['seed']);torch.manual_seed(cfg['seed'])
    torch.use_deterministic_algorithms(True,warn_only=True)
    train_x,train_y,_=load_split('train',data_dir,expected);val_x,val_y,_=load_split('val',data_dir,expected)
    net=CifakeResNet18(pretrained=pretrained and saved is None).to(device)
    groups=[{'params':[p for name,p in net.named_parameters() if not name.startswith('net.fc.')],'lr':cfg['backbone_lr']},
            {'params':list(net.net.fc.parameters()),'lr':cfg['head_lr']}]
    optimizer=torch.optim.AdamW(groups,weight_decay=cfg['weight_decay'])
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=2,min_lr=[1e-6,1e-5])
    run_dir.mkdir(parents=True,exist_ok=True);(run_dir/'checkpoints').mkdir(exist_ok=True)
    epoch,offset,order=1,0,None;history=[];best_loss=float('inf');best_epoch=0;stale=0;early_best=float('inf')
    running_loss,running_correct,seen=0.0,0,0;step=0;prior_seconds=0.0;best_metrics=None
    if saved is not None:
        net.load_state_dict(saved['model']);optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
        for state in optimizer.state.values():
            for key,value in state.items():
                if torch.is_tensor(value) and key!='step':state[key]=value.to(device)
        epoch,offset,order=saved['epoch'],saved['offset'],saved['order']
        history=saved['history'];best_loss=saved['best_loss'];best_epoch=saved['best_epoch'];stale=saved['stale'];early_best=saved['early_best']
        running_loss,running_correct,seen=saved['running_loss'],saved['running_correct'],saved['seen']
        step=saved['step'];prior_seconds=saved['elapsed_seconds'];best_metrics=saved['best_metrics'];restore_rng(saved['rng'],device)
        # An interrupted best-file write can be repaired from the atomically completed epoch file.
        best_file=run_dir/'checkpoints'/f'epoch_{best_epoch:03d}.pt'
        if best_epoch and best_file.exists():
            best_saved=torch.load(best_file,map_location='cpu',weights_only=True)
            atomic_torch(run_dir/'best.pt',{'model':best_saved['model'],'epoch':best_epoch,'validation':best_saved['best_metrics'],
                                           'manifest_sha256':expected,'training':cfg})
    else:
        atomic_json(run_dir/'config.json',dict(training=cfg,manifest_sha256=expected,device=str(device),started_utc=utc_now(),
                    torch=str(torch.__version__),torchvision=str(torchvision.__version__),architecture='ImageNet ResNet18, 512→1 REAL logit',
                    preprocessing='native RGB /255; full-frame bilinear antialiased resize to 224; ImageNet mean/std; no cropping or random augmentation'))
    started=time.monotonic();last_save=started;stop_reason='max_epochs'
    def save_progress(path=None):
        state=dict(model=net.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
                   epoch=epoch,offset=offset,order=order,history=history,best_loss=best_loss,best_epoch=best_epoch,
                   early_best=early_best,stale=stale,best_metrics=best_metrics,running_loss=running_loss,running_correct=running_correct,seen=seen,
                   step=step,rng=rng_state(device),training=cfg,manifest_sha256=expected,device=str(device),
                   elapsed_seconds=prior_seconds+time.monotonic()-started)
        atomic_torch(path or run_dir/'last.pt',state)
    if saved is None:save_progress()
    atomic_json(run_dir/'status.json',dict(status='training',pid=os.getpid(),epoch=epoch,offset=offset,updated_utc=utc_now()))
    safe_to_save=True
    try:
        while epoch<=cfg['epochs']:
            if stale>=cfg['patience'] and epoch>cfg['warmup_epochs']+1:stop_reason='early_stopping';break
            phase=configure_phase(net,epoch,cfg)
            if order is None:order=torch.randperm(len(train_y),generator=torch.Generator().manual_seed(cfg['seed']+epoch)).tolist()
            epoch_started=time.monotonic()
            for start in range(offset,len(order),cfg['batch_size']):
                idx=order[start:start+cfg['batch_size']]
                x=torch.tensor(np.ascontiguousarray(train_x[idx]),device=device,dtype=torch.float32)/255
                y=torch.tensor(train_y[idx],device=device,dtype=torch.float32)
                safe_to_save=False
                optimizer.zero_grad(set_to_none=True);z=net(x);loss=F.binary_cross_entropy_with_logits(z,y)
                if not bool(torch.isfinite(loss)):raise RuntimeError('Non-finite training loss')
                loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1.0);optimizer.step()
                running_loss+=float(loss.detach().cpu())*len(idx);running_correct+=int(((z.detach()>=0)==y.bool()).sum().cpu());seen+=len(idx)
                step+=1;offset=start+len(idx);safe_to_save=True
                if step%50==0:
                    atomic_json(run_dir/'status.json',dict(status='training',pid=os.getpid(),epoch=epoch,phase=phase,offset=offset,
                                train_n=len(train_y),step=step,best_epoch=best_epoch,updated_utc=utc_now()))
                    print(f'Epoch {epoch}/{cfg["epochs"]} {phase}: {offset:,}/{len(train_y):,}; loss {running_loss/seen:.4f}',flush=True)
                if step%cfg['checkpoint_batches']==0 or time.monotonic()-last_save>=300:
                    save_progress();last_save=time.monotonic()
                if max_steps_for_check is not None and step>=max_steps_for_check:
                    save_progress();return {'status':'checkpoint_check','step':step}
                if prior_seconds+time.monotonic()-started>=cfg['max_hours']*3600:
                    save_progress();stop_reason='time_limit';break
            if stop_reason=='time_limit':break
            validation=metrics(val_y,predict(net,val_x,device,cfg['batch_size']))
            row=dict(epoch=epoch,phase=phase,train_loss=running_loss/seen,train_accuracy=running_correct/seen,
                     val_loss=validation['loss'],val_accuracy=validation['accuracy'],val_roc_auc_fake=validation['roc_auc_fake'],
                     backbone_lr=optimizer.param_groups[0]['lr'],head_lr=optimizer.param_groups[1]['lr'],
                     seconds=time.monotonic()-epoch_started)
            history.append(row)
            improved=validation['loss']<best_loss
            if improved:best_loss,best_epoch,best_metrics=validation['loss'],epoch,validation
            if epoch<=cfg['warmup_epochs']:
                stale=0;early_best=validation['loss']
            elif validation['loss']<early_best-cfg['min_delta']:early_best,stale=validation['loss'],0
            else:stale+=1
            if epoch>cfg['warmup_epochs']:scheduler.step(validation['loss'])
            completed=epoch;epoch+=1;offset=0;order=None;running_loss=0.0;running_correct=0;seen=0
            save_progress(run_dir/'checkpoints'/f'epoch_{completed:03d}.pt')
            save_progress()
            if improved:
                atomic_torch(run_dir/'best.pt',dict(model=net.state_dict(),epoch=completed,validation=validation,training=cfg,manifest_sha256=expected))
                # Same state_dict-only convention as Document_Classification.ipynb.
                weight_path=WEIGHTS_PATH if run_dir.resolve()==RUN_DIR.resolve() else run_dir/'model_saves'/'resnet18.pt'
                atomic_torch(weight_path,{k:v.detach().cpu() for k,v in net.state_dict().items()})
            atomic_json(run_dir/'history.json',history)
            print(f'EPOCH {completed} DONE | train {row["train_accuracy"]:.2%} | val {validation["accuracy"]:.2%} loss {validation["loss"]:.5f} | best {best_epoch}',flush=True)
        if best_epoch==0:raise RuntimeError('No full epoch completed; resume from last.pt before evaluating.')
        summary=dict(status='training_completed',best_epoch=best_epoch,completed_epochs=len(history),stop_reason=stop_reason,
                     best_val_loss=best_loss,elapsed_seconds=prior_seconds+time.monotonic()-started,finished_utc=utc_now())
        atomic_json(run_dir/'training_complete.json',summary);atomic_json(run_dir/'status.json',summary)
        return summary
    except BaseException as exc:
        # Only completed optimizer steps are represented in the checkpoint offset.
        if safe_to_save:save_progress()
        atomic_json(run_dir/'status.json',dict(status='interrupted' if isinstance(exc,KeyboardInterrupt) else 'failed',
                    error=str(exc),epoch=epoch,offset=offset,updated_utc=utc_now()))
        raise

In [4]:
PROBE_SEED = 2026

PROBE_FAMILIES = {
    'JPEG': {'values': [95, 80, 60, 40], 'axis': 'JPEG quality',
             'labels': ['q95', 'q80', 'q60', 'q40']},
    'Blur': {'values': [0.3, 0.6, 1.0, 1.5], 'axis': 'Blur standard deviation (pixels)',
             'labels': ['0.3 px', '0.6 px', '1.0 px', '1.5 px']},
    'Resize': {'values': [28, 24, 20, 16], 'axis': 'Intermediate image width (pixels)',
               'labels': ['28 px', '24 px', '20 px', '16 px']},
    'Noise': {'values': [0.01, 0.03, 0.05, 0.10], 'axis': 'Noise standard deviation (RGB in [0, 1])',
              'labels': ['0.01', '0.03', '0.05', '0.10']},
}

def behavior_conditions():
    conditions = [{'family': 'Original', 'value': None, 'label': 'Original', 'level': 0}]
    for family, spec in PROBE_FAMILIES.items():
        for level, (value, label) in enumerate(zip(spec['values'], spec['labels']), start=1):
            conditions.append({'family': family, 'value': value, 'label': label, 'level': level})
    return conditions

def modify_behavior_image(image, family, value, image_index, seed=PROBE_SEED):
    """Pure pixel transformation: uint8 CHW in/out, always 3x32x32."""
    assert image.dtype == np.uint8 and image.shape == (3, 32, 32)
    if family == 'Original':
        return image.copy()
    if family == 'Noise':
        # Reuse one noise field per image at every strength and in the explorer.
        rng = np.random.default_rng(np.random.SeedSequence([seed, int(image_index)]))
        noise = rng.standard_normal(image.shape, dtype=np.float32)
        changed = np.clip(image.astype(np.float32)/255 + float(value)*noise, 0, 1)
        return np.rint(changed*255).astype(np.uint8)
    source = Image.fromarray(image.transpose(1, 2, 0))
    if family == 'JPEG':
        stream = BytesIO()
        # Keep chroma sampling fixed at 4:4:4; vary quality only.
        source.save(stream, format='JPEG', quality=int(value), subsampling=0, optimize=False, progressive=False)
        stream.seek(0)
        with Image.open(stream) as decoded:
            result = np.array(decoded.convert('RGB'), dtype=np.uint8)
    elif family == 'Blur':
        result = np.array(source.filter(ImageFilter.GaussianBlur(radius=float(value))), dtype=np.uint8)
    elif family == 'Resize':
        width = int(value)
        result = np.array(source.resize((width, width), resample=Image.Resampling.BILINEAR)
                         .resize((32, 32), resample=Image.Resampling.BILINEAR), dtype=np.uint8)
    else:
        raise ValueError(f'Unknown modification: {family}')
    return result.transpose(2, 0, 1).copy()

In [5]:
# Saved validation predictions and a small preview set; no training runs here.
BEHAVIOR_METADATA_JSON = '{"split": "val", "n": 10000, "class_counts": {"FAKE": 5000, "REAL": 5000}, "model_state_sha256": "2b73e9f88f029ae7f9d33935864ceeefbd443b7d80a87e532301909a3c0a90a8", "selected_epoch": 19, "manifest_sha256": "098210d89f9441376008d520f72aeef5f65a7aa6b5e0073ba4e875c979e1f205", "seed": 2026, "conditions": [{"family": "Original", "value": null, "label": "Original", "level": 0}, {"family": "JPEG", "value": 95, "label": "q95", "level": 1}, {"family": "JPEG", "value": 80, "label": "q80", "level": 2}, {"family": "JPEG", "value": 60, "label": "q60", "level": 3}, {"family": "JPEG", "value": 40, "label": "q40", "level": 4}, {"family": "Blur", "value": 0.3, "label": "0.3 px", "level": 1}, {"family": "Blur", "value": 0.6, "label": "0.6 px", "level": 2}, {"family": "Blur", "value": 1.0, "label": "1.0 px", "level": 3}, {"family": "Blur", "value": 1.5, "label": "1.5 px", "level": 4}, {"family": "Resize", "value": 28, "label": "28 px", "level": 1}, {"family": "Resize", "value": 24, "label": "24 px", "level": 2}, {"family": "Resize", "value": 20, "label": "20 px", "level": 3}, {"family": "Resize", "value": 16, "label": "16 px", "level": 4}, {"family": "Noise", "value": 0.01, "label": "0.01", "level": 1}, {"family": "Noise", "value": 0.03, "label": "0.03", "level": 2}, {"family": "Noise", "value": 0.05, "label": "0.05", "level": 3}, {"family": "Noise", "value": 0.1, "label": "0.10", "level": 4}], "gallery_indices": {"JPEG": {"FAKE": 752, "REAL": 9255}, "Blur": {"FAKE": 3571, "REAL": 7473}, "Resize": {"FAKE": 1358, "REAL": 7473}, "Noise": {"FAKE": 50, "REAL": 6193}}, "gallery_selection": "For each class and family, choose the largest shift toward the wrong class at level 2 among clean-correct validation images.", "preview_selection": "Gallery cases, largest score recoveries among clean-wrong images at level 2, and six seeded random examples per class; duplicates removed.", "decision_rule": "FAKE if P(FAKE)>0.5; otherwise REAL.", "transforms": {"JPEG": "Re-encode decoded RGB in memory, quality 95/80/60/40; fixed 4:4:4 chroma sampling, no optimization/progressive mode. Original control is not re-encoded.", "Blur": "Pillow GaussianBlur, standard deviation 0.3/0.6/1.0/1.5 pixels.", "Resize": "32 -> 28/24/20/16 -> 32 with Pillow bilinear interpolation in both directions.", "Noise": "Independent Gaussian RGB noise in [0,1], sigma 0.01/0.03/0.05/0.10; clipped to [0,1], rounded to uint8. Same seeded noise field per image across strengths."}, "environment": {"torch": "2.15.0", "numpy": "2.5.4", "pillow": "13.0.0", "jpeg_codec": "6.2", "device": "mps"}, "elapsed_seconds": 10.356811792124063, "scope": "Exploratory validation-set model behavior. Fixed selected model; no training, no test evaluation, no causal claim about learned features."}'
BEHAVIOR_BUNDLE_SHA256 = 'c57176388e6d839f96c1fa443fd1b09079e9c2b72085b1ee8bd978fa1e8046bd'
BEHAVIOR_BUNDLE = (
    '...'
    'BQYAAAAABgAGAHQBAADA+gkAAAA='
)


def load_behavior_bundle():
    raw = base64.b64decode(BEHAVIOR_BUNDLE, validate=True)
    assert hashlib.sha256(raw).hexdigest() == BEHAVIOR_BUNDLE_SHA256
    with np.load(BytesIO(raw), allow_pickle=False) as archive:
        arrays = {key: archive[key].copy() for key in archive.files}
    report = json.loads(BEHAVIOR_METADATA_JSON)
    assert arrays['p_fake'].shape == (len(report['conditions']), report['n'])
    assert len(arrays['labels']) == len(arrays['paths']) == report['n']
    assert np.isfinite(arrays['p_fake']).all()
    assert ((arrays['p_fake'] >= 0) & (arrays['p_fake'] <= 1)).all()
    assert arrays['preview_images'].shape == (len(arrays['preview_indices']), 3, 32, 32)
    assert arrays['preview_images'].dtype == np.uint8
    assert len(np.unique(arrays['preview_indices'])) == len(arrays['preview_indices'])
    assert report['conditions'] == behavior_conditions()
    return arrays, report
BASE_ARRAYS, BASE_REPORT = load_behavior_bundle()

In [6]:
def atomic_npz(path,**arrays):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);temp=p.with_name(p.name+'.tmp')
    with temp.open('wb') as f:np.savez_compressed(f,**arrays)
    temp.replace(p)

def cached_predictions(path,net,images,device,digest):
    p=Path(path)
    if p.exists():
        with np.load(p,allow_pickle=False) as a:
            if str(a['model_digest'])!=digest:raise ValueError('Prediction cache belongs to another model.')
            return a['logits']
    z=predict(net,images,device)
    atomic_npz(p,logits=z,model_digest=np.asarray(digest))
    return z

def ablation_summary(labels,z,removed):
    # removed: channels × images. A positive loss means removal reduces accuracy.
    correct=(z>=0)==labels;after=(removed>=0)==labels[None,:]
    harm=correct[None,:]&~after;repair=~correct[None,:]&after
    rows=[]
    for ch in range(len(removed)):
        h=[int(harm[ch,labels==l].sum()) for l in [0,1]]
        r=[int(repair[ch,labels==l].sum()) for l in [0,1]]
        rows.append([float(correct.mean()-after[ch].mean()),*h,*r,
                     float((removed[ch,labels==0]<0).mean()),float((removed[ch,labels==1]<0).mean())])
    return np.asarray(rows,dtype=np.float64)

@torch.inference_mode()
def internal_probe_resnet(net,images,labels,ids,device,run_dir,digest):
    conditions=[('Original',None),('Resize',28),('JPEG',60)]
    summaries=[[],[]];mean_summaries=[[],[]];zero_summaries=[[],[]];base=[];top_slots=[];top_peaks=[];top_positions=[]
    for ci,(family,value) in enumerate(conditions):
        cache=Path(run_dir)/'probes'/f'internal_{ci}.npz'
        if cache.exists():
            packed=dict(np.load(cache,allow_pickle=False))
            if str(packed['model_digest'])!=digest:raise ValueError('Internal probe cache does not match this checkpoint.')
        else:
            changed=np.stack([modify_behavior_image(im,family,value,int(i)) for im,i in zip(images,ids)])
            n=len(ids);zall=np.empty(n,np.float32);stem_removed=np.empty((64,n),np.float32);last_removed=np.empty((512,n),np.float32)
            mean=[np.zeros(64),np.zeros(512)];zeros=[np.zeros(64),np.zeros(512)]
            peaks=[np.empty((64,n),np.float32),np.empty((512,n),np.float32)]
            positions=[np.empty((64,n,2),np.int16),np.empty((512,n,2),np.int16)]
            for start in range(0,n,64):
                stop=min(n,start+64);x=torch.tensor(changed[start:stop],device=device,dtype=torch.float32)/255
                stem,last,pooled,z=net.forward_maps(x);zall[start:stop]=z.cpu().numpy()
                contribution=pooled*net.net.fc.weight[0]
                reconstructed=contribution.sum(1)+net.net.fc.bias[0]
                torch.testing.assert_close(z,reconstructed,atol=5e-5,rtol=1e-5)
                last_removed[:,start:stop]=(z[:,None]-contribution).cpu().numpy().T
                for ch in range(64):
                    edited=stem.clone();edited[:,ch]=0
                    stem_removed[ch,start:stop]=net.continue_stem(edited).cpu().numpy()
                for li,maps in enumerate([stem,last]):
                    mean[li]+=maps.mean((2,3)).sum(0).cpu().numpy()
                    zeros[li]+=(maps==0).float().mean((2,3)).sum(0).cpu().numpy()
                    if ci==0:
                        p,where=maps.flatten(2).max(2);loc=where.cpu().numpy().T
                        peaks[li][:,start:stop]=p.cpu().numpy().T
                        positions[li][:,start:stop,0]=loc//maps.shape[-1]
                        positions[li][:,start:stop,1]=loc%maps.shape[-1]
                if start==0 or stop==n or stop%512==0:print(f'Internal {family}: {stop}/{n} images, stem and final channels',flush=True)
            packed=dict(model_digest=np.asarray(digest),logits=zall,stem_removed=stem_removed,last_removed=last_removed,
                        stem_mean=mean[0]/n,last_mean=mean[1]/n,stem_zero=zeros[0]/n,last_zero=zeros[1]/n)
            if ci==0:
                for li,name in enumerate(['stem','last']):
                    slots=np.argsort(-peaks[li],axis=1,kind='stable')[:,:3]
                    packed[name+'_top_slots']=slots
                    packed[name+'_top_peaks']=np.take_along_axis(peaks[li],slots,axis=1)
                    packed[name+'_top_positions']=positions[li][np.arange(len(slots))[:,None],slots]
            atomic_npz(cache,**packed)
        base.append(metrics(labels,packed['logits']))
        for li,name in enumerate(['stem','last']):
            summaries[li].append(ablation_summary(labels,packed['logits'],packed[name+'_removed']))
            mean_summaries[li].append(packed[name+'_mean']);zero_summaries[li].append(packed[name+'_zero'])
            if ci==0:
                top_slots.append(packed[name+'_top_slots']);top_peaks.append(packed[name+'_top_peaks']);top_positions.append(packed[name+'_top_positions'])
    result={'cohort_indices':ids,'cohort_labels':labels}
    for li,name in enumerate(['stem','last']):
        result[name+'_ablation']=np.stack(summaries[li]);result[name+'_mean']=np.stack(mean_summaries[li]);result[name+'_zero']=np.stack(zero_summaries[li])
        result[name+'_top_slots']=top_slots[li];result[name+'_top_peaks']=top_peaks[li];result[name+'_top_positions']=top_positions[li]
    return result,base

def finalize_resnet(run_dir=RUN_DIR,data_dir=DATA_DIR,selected_device=None,internal_per_class=1000):
    run_dir=Path(run_dir);data_dir=Path(data_dir);share=run_dir/'share';share.mkdir(parents=True,exist_ok=True)
    if not (run_dir/'training_complete.json').exists():raise ValueError('Finish training and validation selection before final evaluation.')
    device=device_for(DEVICE) if selected_device is None else torch.device(selected_device)
    best=torch.load(run_dir/'best.pt',map_location='cpu',weights_only=True)
    net=CifakeResNet18(pretrained=False).to(device);net.load_state_dict(best['model']);net.eval();digest=model_digest(net)
    val_x,val_y,val_rows=load_split('val',data_dir)
    assert np.array_equal(BASE_ARRAYS['labels'],val_y)
    assert np.array_equal(BASE_ARRAYS['paths'],[r['path'] for r in val_rows])
    atomic_json(run_dir/'status.json',dict(status='evaluating',pid=os.getpid(),best_epoch=best['epoch'],updated_utc=utc_now()))
    val_z=cached_predictions(run_dir/'probes'/'validation.npz',net,val_x,device,digest)
    test_x,test_y,test_rows=load_split('test',data_dir)
    test_z=cached_predictions(run_dir/'probes'/'test.npz',net,test_x,device,digest)
    seen=set();unique=[]
    for i,row in enumerate(test_rows):
        if row['pixel_sha256'] not in seen:seen.add(row['pixel_sha256']);unique.append(i)
    evaluation=dict(validation=metrics(val_y,val_z),official_test=metrics(test_y,test_z),unique_test=metrics(test_y[unique],test_z[unique]),
                    selected_epoch=best['epoch'],model_digest=digest,manifest_sha256=MANIFEST_SHA256)
    atomic_json(run_dir/'test_metrics.json',evaluation)
    print('Selected epoch',best['epoch'],'clean validation',evaluation['validation']['accuracy'],'official test',evaluation['official_test']['accuracy'],flush=True)
    # Plain state_dict: instantiate CifakeResNet18(False), load_state_dict(...), then eval().
    atomic_torch(share/'resnet18_plain_seed1.pt',{k:v.detach().cpu() for k,v in net.state_dict().items()})
    conditions=behavior_conditions();scores=[];condition_metrics=[]
    for ci,condition in enumerate(conditions):
        atomic_json(run_dir/'status.json',dict(status='validation_robustness',condition=ci,total=len(conditions),pid=os.getpid(),updated_utc=utc_now()))
        changed=val_x if ci==0 else np.stack([modify_behavior_image(im,condition['family'],condition['value'],i) for i,im in enumerate(val_x)])
        z=cached_predictions(run_dir/'probes'/f'condition_{ci:02d}.npz',net,changed,device,digest)
        scores.append(z);condition_metrics.append(metrics(val_y,z))
        print(f'Validation {condition["family"]} {condition["label"]}: {condition_metrics[-1]["accuracy"]:.2%}',flush=True)
    val_logits=np.stack(scores)
    preview_ids=list(map(int,BASE_ARRAYS['preview_indices']))
    original_correct=(val_logits[0]>=0)==val_y
    gallery={}
    for family in PROBE_FAMILIES:
        ci=next(i for i,c in enumerate(conditions) if c['family']==family and c['level']==2)
        gallery[family]={}
        for label,name in [(0,'FAKE'),(1,'REAL')]:
            candidates=np.flatnonzero((val_y==label)&original_correct)
            if not len(candidates):candidates=np.flatnonzero(val_y==label)
            direction=1 if label==0 else -1
            i=int(candidates[np.argmax(direction*(val_logits[ci,candidates]-val_logits[0,candidates]))])
            gallery[family][name]=i;preview_ids.append(i)
    preview_ids=np.asarray(list(dict.fromkeys(preview_ids)),dtype=np.int32)
    rng=np.random.default_rng(20261103)
    cohort=np.sort(np.concatenate([rng.choice(np.flatnonzero(val_y==l),internal_per_class,replace=False) for l in [0,1]])).astype(np.int32)
    atomic_json(run_dir/'status.json',dict(status='internal_analysis',pid=os.getpid(),n=len(cohort),updated_utc=utc_now()))
    internal,internal_base=internal_probe_resnet(net,val_x[cohort],val_y[cohort],cohort,device,run_dir,digest)
    example_ids=np.unique(np.concatenate([cohort[internal['stem_top_slots']].ravel(),cohort[internal['last_top_slots']].ravel()])).astype(np.int32)
    arrays=dict(val_labels=val_y,val_logits=val_logits,val_paths=np.asarray([r['path'] for r in val_rows]),
                preview_indices=preview_ids,preview_images=val_x[preview_ids],
                example_indices=example_ids,example_images=val_x[example_ids],**internal)
    atomic_npz(share/'resnet18_analysis.npz',**arrays)
    report=dict(model_digest=digest,manifest_sha256=MANIFEST_SHA256,selected_epoch=best['epoch'],conditions=conditions,
                condition_metrics=condition_metrics,evaluation=evaluation,gallery_indices=gallery,
                internal_conditions=['Original','Resize 32→28→32','JPEG quality 60'],internal_baseline_metrics=internal_base,
                internal_summary_columns=['accuracy_lost','new_errors_fake','new_errors_real','repaired_fake','repaired_real','fake_detected','real_false_alarm'],
                history=json.loads((run_dir/'history.json').read_text()),training=best['training'],created_utc=utc_now())
    atomic_json(share/'resnet18_report.json',report)
    manifest=dict(format_version=1,run_name=RUN_NAME,selected_epoch=best['epoch'],model_digest=digest,created_utc=utc_now(),
                  assets={name:sha256(share/name) for name in ['resnet18_plain_seed1.pt','resnet18_analysis.npz','resnet18_report.json']})
    atomic_json(share/'resnet18_manifest.json',manifest)
    return report

def load_review_assets(download=DOWNLOAD_SHARED_MODEL,share_dir=SHARE_DIR,selected_device=None):
    share=Path(share_dir);manifest_path=share/'resnet18_manifest.json'
    if not manifest_path.exists() and download:
        base=f'https://github.com/{REPOSITORY}/releases/download/{RELEASE_TAG}/'
        try:
            with urllib.request.urlopen(base+'resnet18_manifest.json',timeout=20) as response:raw=response.read()
        except urllib.error.HTTPError as error:
            if error.code==404:
                print('The finished ResNet18 package has not been published yet. Review mode does not train a model.');return None,None,None
            raise
        share.mkdir(parents=True,exist_ok=True)
        manifest=json.loads(raw)
        for name,expected in manifest['assets'].items():
            if Path(name).name!=name:raise ValueError('Invalid asset filename')
            target=share/name;temporary=target.with_suffix(target.suffix+'.download')
            urllib.request.urlretrieve(base+name,temporary)
            if sha256(temporary)!=expected:raise ValueError('Downloaded asset failed its checksum.')
            temporary.replace(target)
        manifest_path.write_bytes(raw)
    if not manifest_path.exists():
        print('No finished model package is available locally yet.');return None,None,None
    manifest=json.loads(manifest_path.read_text())
    for name,expected in manifest['assets'].items():
        if sha256(share/name)!=expected:raise ValueError(f'Asset checksum mismatch: {name}')
    state_dict=torch.load(share/'resnet18_plain_seed1.pt',map_location='cpu',weights_only=True)
    device=device_for(DEVICE) if selected_device is None else torch.device(selected_device)
    net=CifakeResNet18(pretrained=False).to(device);net.load_state_dict(state_dict);net.eval()
    if model_digest(net)!=manifest['model_digest']:raise ValueError('Loaded weights do not match the published model.')
    arrays=dict(np.load(share/'resnet18_analysis.npz',allow_pickle=False));report=json.loads((share/'resnet18_report.json').read_text())
    assert report['model_digest']==manifest['model_digest'] and report['manifest_sha256']==MANIFEST_SHA256
    return net,arrays,report

In [7]:
resnet_pair_cache={}
resnet_occlusion_cache={}

def finish_plot(fig,name):
    if globals().get('SAVE_FIGURES',False):
        destination=RUN_DIR/'figures';destination.mkdir(parents=True,exist_ok=True)
        fig.savefig(destination/f'{name}.png',dpi=150,bbox_inches='tight')
    plt.show();plt.close(fig)
def picture(ax,image,title):
    ax.imshow(image.transpose(1,2,0),interpolation='nearest');ax.set_title(title,fontsize=9);ax.set(xticks=[],yticks=[])
def resnet_pair(index,family,level):
    key=(resnet_report['model_digest'],int(index),family,int(level))
    if key in resnet_pair_cache:return resnet_pair_cache[key]
    conditions=resnet_report['conditions'];indices=[0]+[i for i,c in enumerate(conditions) if c['family']==family]
    condition=conditions[indices[int(level)]]
    original=resnet_previews[int(index)]
    changed=modify_behavior_image(original,condition['family'],condition['value'],int(index))
    images=np.stack([original,changed]);resnet_model.eval()
    with torch.inference_mode(False),torch.enable_grad():
        x=torch.tensor(images,device=device,dtype=torch.float32)/255
        stem,last,pooled,z=resnet_model.forward_maps(x)
        gradient=torch.autograd.grad((-z).sum(),last)[0]
        cam=torch.relu((gradient.mean((2,3),keepdim=True)*last).sum(1))
        contributions=pooled*resnet_model.net.fc.weight[0]
        bias=float(resnet_model.net.fc.bias[0].detach().cpu())
        values=dict(stem=stem.detach().cpu().numpy(),last=last.detach().cpu().numpy(),cam=cam.detach().cpu().numpy(),
                    contributions=contributions.detach().cpu().numpy(),pooled=pooled.detach().cpu().numpy(),
                    logits=z.detach().cpu().numpy(),bias=bias)
    np.testing.assert_allclose(values['contributions'].sum(1)+bias,values['logits'],atol=5e-5,rtol=1e-5)
    pair=dict(index=int(index),images=images,values=values,family=family,level=int(level),
              setting='Original' if level==0 else f'{family} {condition["label"]}',
              truth='FAKE' if resnet_data['val_labels'][index]==0 else 'REAL',path=str(resnet_data['val_paths'][index]))
    if len(resnet_pair_cache)>=16:resnet_pair_cache.pop(next(iter(resnet_pair_cache)))
    resnet_pair_cache[key]=pair;return pair

def learning_figure():
    h=resnet_report['history'];epochs=[r['epoch'] for r in h]
    fig,axes=plt.subplots(1,2,figsize=(11,3.8),layout='constrained')
    for ax,key,title in zip(axes,['loss','accuracy'],['Binary cross entropy','Accuracy']):
        ax.plot(epochs,[r['train_'+key] for r in h],label='Train');ax.plot(epochs,[r['val_'+key] for r in h],label='Validation')
        ax.axvline(resnet_report['selected_epoch'],color='black',ls='--',lw=1,label='Selected epoch')
        ax.axvspan(.5,resnet_report['training']['warmup_epochs']+.5,color='#dddddd',alpha=.4)
        ax.set(xlabel='Epoch',ylabel=title);ax.legend(fontsize=8);ax.grid(alpha=.2)
    return fig

def robustness_figure():
    fig,axes=plt.subplots(3,4,figsize=(14,9),layout='constrained')
    base_z=-np.log(np.clip(BASE_ARRAYS['p_fake'],1e-7,1-1e-7)/(1-np.clip(BASE_ARRAYS['p_fake'],1e-7,1-1e-7)))
    base_metrics=[metrics(BASE_ARRAYS['labels'],z) for z in base_z]
    for col,family in enumerate(PROBE_FAMILIES):
        indices=[0]+[i for i,c in enumerate(resnet_report['conditions']) if c['family']==family]
        for row,(key,label) in enumerate([('accuracy','Accuracy'),('fake_detection_rate','Fake images detected'),('real_flagged_fake_rate','Real false alarms')]):
            ax=axes[row,col]
            ax.plot(range(5),[resnet_report['condition_metrics'][i][key] for i in indices],marker='o',label='ResNet18')
            ax.plot(range(5),[base_metrics[i][key] for i in indices],marker='s',ls='--',label='Plain CNN')
            ax.set(xticks=range(5),xticklabels=['Original']+PROBE_FAMILIES[family]['labels'],ylim=(-.03,1.03),ylabel=label)
            if row==0:ax.set_title(family)
            ax.tick_params(axis='x',labelrotation=25,labelsize=8);ax.grid(alpha=.2)
    axes[0,0].legend();fig.suptitle('Same 10,000 validation images and native 32×32 edits · frozen selected models')
    return fig

def confusion_figure():
    cm=np.asarray(resnet_report['evaluation']['official_test']['confusion_matrix'])
    fig,ax=plt.subplots(figsize=(5,4),layout='constrained');ax.imshow(cm,cmap='Blues')
    for y in range(2):
        for x in range(2):ax.text(x,y,f'{cm[y,x]:,}',ha='center',va='center',color='white' if cm[y,x]>cm.max()/2 else 'black')
    ax.set(xticks=[0,1],xticklabels=['FAKE','REAL'],yticks=[0,1],yticklabels=['FAKE','REAL'],xlabel='Predicted',ylabel='True',title='Official test · selected ResNet18')
    return fig

def gradcam_figure(pair):
    fig,axes=plt.subplots(2,3,figsize=(11,7),layout='constrained');cam=pair['values']['cam'];vmax=max(float(cam.max()),1e-9)
    for side in range(2):
        name='Original' if side==0 else pair['setting'];p=pfake(pair['values']['logits'][side])
        picture(axes[side,0],pair['images'][side],f'{name} · P(FAKE) {p:.2%}')
        heat=axes[side,1].imshow(cam[side],cmap='magma',vmin=0,vmax=vmax,interpolation='nearest')
        axes[side,1].set(title='Native FAKE Grad-CAM · 7×7',xticks=[],yticks=[])
        picture(axes[side,2],pair['images'][side],'Overlay on native input')
        resized=F.interpolate(torch.tensor(cam[side])[None,None],size=(32,32),mode='bilinear',align_corners=False)[0,0].numpy()
        axes[side,2].imshow(np.ma.masked_where(resized<=0,resized),cmap='magma',vmin=0,vmax=vmax,alpha=.55)
    fig.colorbar(heat,ax=list(axes[:,1:].flat),shrink=.7,label='Positive FAKE Grad-CAM · shared scale')
    fig.suptitle(f"Target fixed at FAKE (−REAL logit) · true label {pair['truth']}\n{pair['path']}")
    return fig

def contribution_figure(pair):
    c=pair['values']['contributions'].astype(np.float64);delta=c[1]-c[0];order=np.argsort(-np.abs(delta),kind='stable')[:12]
    other=np.ones(512,bool);other[order]=False;bars=np.r_[delta[order],delta[other].sum()]
    fig,axes=plt.subplots(2,1,figsize=(12,7),layout='constrained',gridspec_kw={'height_ratios':[1,3]})
    vmax=max(float(np.abs(c).max()),1e-9);heat=axes[0].imshow(c,aspect='auto',cmap='RdBu',vmin=-vmax,vmax=vmax)
    axes[0].set(yticks=[0,1],yticklabels=['Original',pair['setting']],xlabel='Final feature channel (0–511)',title='All 512 pooled-activation × output-weight contributions')
    fig.colorbar(heat,ax=axes[0],shrink=.8,label='Contribution to REAL logit')
    axes[1].barh(range(13),bars,color=['#0072B2' if x>=0 else '#D55E00' for x in bars])
    axes[1].set(yticks=range(13),yticklabels=[f'Channel {i}' for i in order]+['Other 500 channels'],xlabel='Change in REAL logit contribution')
    axes[1].invert_yaxis();axes[1].axvline(0,color='black',lw=.8)
    z=pair['values']['logits'];fig.suptitle(f'Exact final-layer accounting · REAL logit {z[0]:+.3f} → {z[1]:+.3f} · fixed bias {pair["values"]["bias"]:+.4f}')
    return fig

@torch.inference_mode()
def native_occlusion(image,reference,patch=4):
    stride=patch//2;starts=list(range(0,33-patch,stride));positions=[(y,x) for y in starts for x in starts]
    x=torch.tensor(image,device=device,dtype=torch.float32)[None]/255
    original=float(resnet_model(x).cpu()[0]);drops=[];fill=torch.tensor(reference,device=device,dtype=torch.float32)
    for start in range(0,len(positions),64):
        current=positions[start:start+64];batch=x.expand(len(current),-1,-1,-1).clone()
        for i,(y,xpos) in enumerate(current):batch[i,:,y:y+patch,xpos:xpos+patch]=fill[:,None,None]
        drops.extend((resnet_model(batch)-original).cpu().numpy().tolist())
    sums=np.zeros((32,32));coverage=np.zeros((32,32))
    for (y,xpos),drop in zip(positions,drops):sums[y:y+patch,xpos:xpos+patch]+=drop;coverage[y:y+patch,xpos:xpos+patch]+=1
    assert np.all(coverage>0)
    return sums/coverage

def occlusion_figure(pair,patch=4):
    reference_names=['Original mean RGB','Mid-grey RGB'];references=[pair['images'][0].mean((1,2))/255,np.full(3,.5)]
    maps=[]
    for side in range(2):
        row=[]
        for reference in references:
            key=(resnet_report['model_digest'],pair['index'],pair['family'],pair['level'],side,reference.tobytes(),patch)
            if key not in resnet_occlusion_cache:resnet_occlusion_cache[key]=native_occlusion(pair['images'][side],reference,patch)
            row.append(resnet_occlusion_cache[key])
        maps.append(row)
    vmax=max(float(np.abs(maps).max()),1e-9);fig,axes=plt.subplots(2,3,figsize=(11,7),layout='constrained')
    for side in range(2):
        picture(axes[side,0],pair['images'][side],('Original' if side==0 else pair['setting'])+f" · P(FAKE) {pfake(pair['values']['logits'][side]):.2%}")
        for j in range(2):
            heat=axes[side,j+1].imshow(maps[side][j],cmap='RdBu_r',vmin=-vmax,vmax=vmax,interpolation='nearest')
            axes[side,j+1].set(title=reference_names[j],xticks=[],yticks=[])
    fig.colorbar(heat,ax=list(axes[:,1:].flat),shrink=.7,label='FAKE score before − after covering · red positive, blue negative')
    fig.suptitle(f'Occlusion at native 32×32 · {patch}×{patch} patch · stride {patch//2}\nSame reference within each pair; one colour scale across all maps')
    return fig

def channel_figure(pair,stage,channel):
    maps=pair['values'][stage][:,channel];vmax=max(float(maps.max()),1e-9)
    fig,axes=plt.subplots(2,2,figsize=(8,7),layout='constrained')
    with torch.inference_mode():
        if stage=='last':removed=pair['values']['logits']-pair['values']['contributions'][:,channel]
        else:
            stem=torch.tensor(pair['values']['stem'],device=device);stem[:,channel]=0
            removed=resnet_model.continue_stem(stem).cpu().numpy()
    for side in range(2):
        picture(axes[side,0],pair['images'][side],('Original' if side==0 else pair['setting'])+
                f"\nP(FAKE) {pfake(pair['values']['logits'][side]):.2%}; channel zeroed {pfake(removed[side]):.2%}")
        heat=axes[side,1].imshow(maps[side],cmap='magma',vmin=0,vmax=vmax,interpolation='nearest')
        axes[side,1].set(title=f'{stage} channel {channel} · {maps.shape[-1]}×{maps.shape[-1]}',xticks=[],yticks=[])
    fig.colorbar(heat,ax=list(axes[:,1]),shrink=.8,label='Post-ReLU activation · common scale')
    fig.suptitle('Stem = Conv–BatchNorm–ReLU; last = layer4 output after residual additions')
    return fig

def weight_figure(stage='stem',channel=0):
    if stage=='last':
        weights=resnet_model.net.fc.weight[0].detach().cpu().numpy();fig,ax=plt.subplots(figsize=(12,4),layout='constrained')
        ax.bar(range(512),weights,color='#0072B2');ax.scatter([channel],[weights[channel]],color='#D55E00',zorder=5)
        ax.set(xlabel='Final feature channel',ylabel='Weight into REAL logit',title=f'Final linear weights · selected channel {channel} = {weights[channel]:+.4f}')
        return fig
    weights=resnet_model.net.conv1.weight.detach().cpu().numpy();vmax=float(np.abs(weights).max())
    fig,axes=plt.subplots(8,8,figsize=(14,10),layout='constrained')
    for i,ax in enumerate(axes.flat):
        heat=ax.imshow(np.concatenate(list(weights[i]),axis=1),cmap='RdBu_r',vmin=-vmax,vmax=vmax,interpolation='nearest')
        ax.set(title=str(i),xticks=[3,10,17],xticklabels=['R','G','B'],yticks=[]);ax.tick_params(labelsize=7)
        if i==channel:
            for spine in ax.spines.values():spine.set_color('green');spine.set_linewidth(2)
    fig.colorbar(heat,ax=list(axes.flat),shrink=.8,label='Learned convolution weight · shared scale')
    fig.suptitle('ResNet18 stem: 64 filters, each with three 7×7 RGB slices · BatchNorm and ReLU follow this convolution')
    return fig

def examples_figure(stage,channel):
    slots=resnet_data[stage+'_top_slots'][channel];ids=resnet_data['cohort_indices'][slots]
    images=np.stack([resnet_examples[int(i)] for i in ids])
    with torch.inference_mode():
        stem,last,_,z=resnet_model.forward_maps(torch.tensor(images,device=device,dtype=torch.float32)/255)
        maps=(stem if stage=='stem' else last)[:,channel].cpu().numpy()
    fig,axes=plt.subplots(3,2,figsize=(7,9),layout='constrained');vmax=max(float(maps.max()),1e-9)
    for i,index in enumerate(ids):
        truth='FAKE' if resnet_data['val_labels'][index]==0 else 'REAL'
        picture(axes[i,0],images[i],f'{truth} · {Path(str(resnet_data["val_paths"][index])).name}')
        heat=axes[i,1].imshow(maps[i],cmap='magma',vmin=0,vmax=vmax,interpolation='nearest')
        axes[i,1].set(title=f'Peak activation {maps[i].max():.3f}',xticks=[],yticks=[])
    fig.colorbar(heat,ax=list(axes[:,1]),shrink=.7,label='Activation')
    fig.suptitle(f'{stage} channel {channel}: three strongest spatial responses in the fixed validation sample\nThese examples suggest response patterns, not semantic class labels.')
    return fig

def internal_figure():
    fig,axes=plt.subplots(1,2,figsize=(13,7),layout='constrained')
    for ax,stage in zip(axes,['stem','last']):
        values=resnet_data[stage+'_ablation'][:,:,0]*100;chosen=np.argsort(-np.abs(values[0]),kind='stable')[:20]
        for ci,name in enumerate(resnet_report['internal_conditions']):ax.plot(values[ci,chosen],range(20),marker='o',label=name)
        ax.set(yticks=range(20),yticklabels=[str(i) for i in chosen],xlabel='Accuracy lost when channel is zeroed (pp)',ylabel='Channel',title=f'{stage}: 20 largest clean effects')
        ax.invert_yaxis();ax.axvline(0,color='black',lw=.8);ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.suptitle(f"Paired channel removal on the same {len(resnet_data['cohort_indices']):,} validation images\nPositive = removal hurts accuracy; negative = removal improves accuracy. All channels are available in the explorer.")
    return fig

def draw_explorer():
    import ipywidgets as widgets
    import html
    image_choice=widgets.Dropdown(options=[(f"{'FAKE' if resnet_data['val_labels'][int(i)]==0 else 'REAL'} · {resnet_data['val_paths'][int(i)]}",int(i)) for i in resnet_data['preview_indices']],
                                   value=default_pair['index'],description='Image:',layout=widgets.Layout(width='95%'))
    family_choice=widgets.Dropdown(options=list(PROBE_FAMILIES),value='Resize',description='Change:')
    level_choice=widgets.IntSlider(value=1,min=0,max=4,description='Strength:',continuous_update=False)
    stage_choice=widgets.Dropdown(options=[('Stem', 'stem'),('Final features','last')],value='stem',description='Stage:')
    channel_choice=widgets.IntSlider(value=default_stem_channel,min=0,max=63,description='Channel:',continuous_update=False)
    patch_choice=widgets.Dropdown(options=[4,8],value=4,description='Patch:')
    button=widgets.Button(description='Inspect',button_style='primary');status=widgets.HTML();details=widgets.HTML()
    pictures=[widgets.Image(format='png',layout=widgets.Layout(width='100%')) for _ in range(7)];stats=widgets.HTML()
    tabs=widgets.Tab(children=[widgets.VBox(pictures[:2]),pictures[2],pictures[3],pictures[4],pictures[5],pictures[6],stats])
    for i,title in enumerate(['Attribution','Occlusion','Channel','Weights','Examples','All effects','Selected effects']):tabs.set_title(i,title)
    state={}
    def stale(change=None):status.value='Settings changed; press Inspect. The current plots still show the last completed selection.'
    def stage_changed(change):
        channel_choice.max=63 if stage_choice.value=='stem' else 511
        channel_choice.value=min(channel_choice.value,channel_choice.max);stale()
    def update(_=None):
        button.disabled=True;status.value='Computing explanations…'
        try:
            pair=resnet_pair(image_choice.value,family_choice.value,level_choice.value);stage=stage_choice.value;ch=channel_choice.value
            draws=[lambda:gradcam_figure(pair),lambda:contribution_figure(pair),lambda:occlusion_figure(pair,patch_choice.value),
                   lambda:channel_figure(pair,stage,ch),lambda:weight_figure(stage,ch),lambda:examples_figure(stage,ch),internal_figure]
            rendered=[]
            for draw in draws:
                fig=draw();stream=BytesIO()
                try:fig.savefig(stream,format='png',dpi=105,bbox_inches='tight');rendered.append(stream.getvalue())
                finally:plt.close(fig)
            for target,png in zip(pictures,rendered):target.value=png
            rows=[]
            for ci,name in enumerate(resnet_report['internal_conditions']):
                v=resnet_data[stage+'_ablation'][ci,ch];rows.append(f'<tr><td>{name}</td><td>{v[0]*100:+.2f}</td><td>{int(v[1]+v[2])}</td><td>{int(v[3]+v[4])}</td><td>{v[5]:.1%}</td><td>{v[6]:.1%}</td></tr>')
            stats.value='<table><tr><th>Condition</th><th>Accuracy lost (pp)</th><th>New errors</th><th>Repaired errors</th><th>Fake detection after removal</th><th>Real false alarms after removal</th></tr>'+''.join(rows)+'</table>'
            details.value=f"<b>{stage} channel {ch}</b> · {html.escape(pair['path'])} · {pair['setting']} · true label {pair['truth']}"
            state.update(index=pair['index'],stage=stage,channel=ch,level=pair['level'],family=pair['family'],pair=pair)
            status.value='Ready. All views match the selection recorded below; aggregate effects use the three fixed validation conditions.'
        except Exception as error:status.value='Could not update: '+html.escape(str(error));raise
        finally:button.disabled=False
    for control in [image_choice,family_choice,level_choice,channel_choice,patch_choice]:control.observe(stale,names='value')
    stage_choice.observe(stage_changed,names='value');button.on_click(update);update()
    display(widgets.VBox([image_choice,widgets.HBox([family_choice,level_choice]),widgets.HBox([stage_choice,channel_choice,patch_choice,button]),status,details,tabs]))
    return dict(image=image_choice,family=family_choice,level=level_choice,stage=stage_choice,channel=channel_choice,patch=patch_choice,button=button,status=status,pictures=pictures,state=state)

In [8]:
@contextmanager
def run_lock(run_dir=RUN_DIR):
    directory=Path(run_dir);directory.mkdir(parents=True,exist_ok=True)
    with (directory/'run.lock').open('a+') as handle:
        try:fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('Another process is already using this run.')
        try:yield
        finally:fcntl.flock(handle,fcntl.LOCK_UN)

def render_review_notebook(run_dir=RUN_DIR):
    import nbformat
    from nbclient import NotebookClient
    directory=Path(run_dir);source=directory/'source'/'resnet18_finetuning.ipynb'
    if not source.exists():source=NOTEBOOK_PATH
    nb=nbformat.read(source,as_version=4)
    import re
    for cell in nb.cells:
        if cell.cell_type=='code' and cell.id=='resnet-config':
            cell.source=re.sub(r'^MODE=.*$',"MODE='review'",cell.source,flags=re.M)
            cell.source=re.sub(r'^START_BACKGROUND=.*$','START_BACKGROUND=False',cell.source,flags=re.M)
            cell.source=re.sub(r'^DOWNLOAD_SHARED_MODEL=.*$','DOWNLOAD_SHARED_MODEL=False',cell.source,flags=re.M)
    nb=NotebookClient(nb,timeout=600,kernel_name='cifake',allow_errors=False,store_widget_state=False,
                      resources={'metadata':{'path':str(ROOT)}}).execute()
    nb.metadata.pop('widgets',None)
    for cell in nb.cells:
        if cell.cell_type=='code':
            cell.outputs=[o for o in cell.outputs if not (o.output_type in {'display_data','execute_result'} and 'application/vnd.jupyter.widget-view+json' in o.get('data',{}))]
            if cell.id=='resnet-config':cell.source=cell.source.replace('DOWNLOAD_SHARED_MODEL=False','DOWNLOAD_SHARED_MODEL=True')
    destination=directory/'share'/'resnet18_finetuning.ipynb';nbformat.validate(nb);nbformat.write(nb,destination)
    return destination

def export_team_bundle(run_dir=RUN_DIR):
    directory=Path(run_dir);share=directory/'share';path=share/'resnet18_team_bundle.zip'
    with zipfile.ZipFile(path.with_suffix('.tmp'),mode='w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
        archive.write(share/'resnet18_finetuning.ipynb','resnet18_finetuning.ipynb')
        for filename in ['resnet18_plain_seed1.pt','resnet18_analysis.npz','resnet18_report.json','resnet18_manifest.json']:
            archive.write(share/filename,f'runs/{RUN_NAME}/share/{filename}')
        archive.write(share/'resnet18_plain_seed1.pt',f'model_saves/{RUN_NAME}.pt')
    path.with_suffix('.tmp').replace(path)
    return path

def publish_model_package(run_dir=RUN_DIR):
    directory=Path(run_dir);share=directory/'share';manifest=json.loads((share/'resnet18_manifest.json').read_text())
    digest=manifest['model_digest'];report=json.loads((share/'resnet18_report.json').read_text())
    query=subprocess.run(['gh','release','view',RELEASE_TAG,'--repo',REPOSITORY,'--json','body'],text=True,capture_output=True)
    marker=f'Model fingerprint: {digest}'
    if query.returncode==0:
        if marker not in json.loads(query.stdout)['body']:raise RuntimeError('Release tag already belongs to another model; choose a new tag.')
    else:
        notes=share/'release_notes.txt'
        notes.write_text(f'CIFAKE ResNet18, selected epoch {report["selected_epoch"]}.\n\n'
                         f'Validation accuracy: {report["evaluation"]["validation"]["accuracy"]:.2%}. '
                         f'Official clean test accuracy: {report["evaluation"]["official_test"]["accuracy"]:.2%}.\n\n'
                         'Download resnet18_team_bundle.zip and run resnet18_finetuning.ipynb in review mode. '
                         'The package contains the trained state_dict, saved evaluation/validation probes and an executed notebook. No fine-tuning or dataset download is needed for review.\n\n'
                         'The .pt asset is a plain PyTorch state_dict, following the model_saves convention. Full optimizer checkpoints remain local.\n\n'+marker+'\n')
        launch=json.loads((directory/'source'/'launch.json').read_text())
        subprocess.run(['gh','release','create',RELEASE_TAG,'--repo',REPOSITORY,'--target',launch['git_commit'],
                        '--title','CIFAKE ResNet18 · trained model and review notebook','--notes-file',str(notes),'--latest=false'],check=True)
    assets=[share/name for name in ['resnet18_plain_seed1.pt','resnet18_analysis.npz','resnet18_report.json','resnet18_manifest.json',
                                    'resnet18_finetuning.ipynb','resnet18_team_bundle.zip']]
    subprocess.run(['gh','release','upload',RELEASE_TAG,'--repo',REPOSITORY,*map(str,assets),'--clobber'],check=True)
    return f'https://github.com/{REPOSITORY}/releases/tag/{RELEASE_TAG}'

def run_pipeline(mode='train',publish=PUBLISH_TO_GITHUB):
    with run_lock():
        try:
            if mode in {'train','resume'}:train_resnet(resume=mode=='resume')
            elif mode!='finalize':raise ValueError('Choose train, resume or finalize.')
            report=finalize_resnet()
            # Refresh the simple, user-facing weights file from the selected checkpoint.
            atomic_torch(WEIGHTS_PATH,torch.load(SHARE_DIR/'resnet18_plain_seed1.pt',map_location='cpu',weights_only=True))
            atomic_json(RUN_DIR/'status.json',dict(status='rendering_review',pid=os.getpid(),updated_utc=utc_now()))
            if torch.backends.mps.is_available():torch.mps.empty_cache()
            render_review_notebook();bundle=export_team_bundle()
            atomic_json(RUN_DIR/'status.json',dict(status='publishing' if publish else 'completed',pid=os.getpid(),updated_utc=utc_now()))
            url=publish_model_package() if publish else None
            status=dict(status='completed',selected_epoch=report['selected_epoch'],validation_accuracy=report['evaluation']['validation']['accuracy'],
                        test_accuracy=report['evaluation']['official_test']['accuracy'],team_bundle=str(bundle),release_url=url,finished_utc=utc_now())
            atomic_json(RUN_DIR/'status.json',status);print(json.dumps(status,indent=2),flush=True)
        except BaseException as error:
            atomic_json(RUN_DIR/'status.json',dict(status='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',
                        error=str(error),updated_utc=utc_now(),recovery='Use resume if training_complete.json is absent; otherwise use finalize.'))
            raise

def launch_background(mode='train'):
    import nbformat
    if mode not in {'train','resume','finalize'}:raise ValueError('Invalid background mode')
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    status_path=RUN_DIR/'status.json'
    if status_path.exists():
        status=json.loads(status_path.read_text())
        if status.get('pid') and status.get('status') not in {'completed','failed','interrupted'}:
            try:os.kill(status['pid'],0)
            except ProcessLookupError:pass
            else:raise RuntimeError(f'A process is already active: PID {status["pid"]}')
    if mode=='train' and (RUN_DIR/'last.pt').exists():raise ValueError('This run exists. Choose resume.')
    source=RUN_DIR/'source';source.mkdir(exist_ok=True)
    snapshot=source/'resnet18_finetuning.ipynb'
    if not snapshot.exists():shutil.copy2(NOTEBOOK_PATH,snapshot)
    nb=nbformat.read(snapshot,as_version=4)
    code='\n\n'.join(c.source for c in nb.cells if c.cell_type=='code' and 'runtime' in c.metadata.get('tags',[]))
    runner=source/'runner.py'
    if not runner.exists():runner.write_text(code+'\n\nif __name__ == "__main__":\n    run_pipeline(sys.argv[1] if len(sys.argv)>1 else "train")\n')
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
    atomic_json(source/'launch.json',dict(git_commit=commit,notebook_sha256=sha256(snapshot),runner_sha256=sha256(runner),created_utc=utc_now()))
    env=dict(os.environ);env['PYTHONUNBUFFERED']='1';env.setdefault('MPLCONFIGDIR',str(RUN_DIR/'runtime'/'matplotlib'))
    for key,folder in [('IPYTHONDIR','ipython'),('JUPYTER_CONFIG_DIR','jupyter-config'),('JUPYTER_RUNTIME_DIR','jupyter-runtime')]:
        target=RUN_DIR/'runtime'/folder;target.mkdir(parents=True,exist_ok=True);env[key]=str(target)
    with (RUN_DIR/'training.log').open('ab',buffering=0) as log:
        process=subprocess.Popen([sys.executable,'-u',str(runner),mode],cwd=ROOT,env=env,stdin=subprocess.DEVNULL,
                                 stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    awake=None
    if sys.platform=='darwin' and shutil.which('caffeinate'):
        awake=subprocess.Popen(['caffeinate','-i','-w',str(process.pid)],stdin=subprocess.DEVNULL,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,start_new_session=True).pid
    atomic_json(RUN_DIR/'launch.json',dict(pid=process.pid,caffeinate_pid=awake,mode=mode,started_utc=utc_now(),log=str(RUN_DIR/'training.log')))
    print(f'Background {mode} launched: PID {process.pid}. Log: {RUN_DIR / "training.log"}')
    return process.pid

## 3. Start, monitor and recover the run

The saving convention follows [Document_Classification.ipynb](https://github.com/adamp-22/document-classification-nlp/blob/main/Document_Classification.ipynb): the selected model's plain `state_dict()` is saved to **`model_saves/resnet18_plain_seed1.pt`**, and a fresh model instance can load it for inference. Here, selection uses validation loss, matching our baseline protocol.

For recovery we additionally save `runs/resnet18_plain_seed1/last.pt` every 200 batches or five minutes, whichever comes first, and after every epoch. It contains the model, optimizer, scheduler, random states, shuffled order, next batch offset and training history. `checkpoints/epoch_XXX.pt` retains full epoch snapshots; `best.pt` retains the selected model and its metadata. Writes use a temporary file and atomic replacement. A failure in the middle of an optimizer update preserves the preceding valid checkpoint rather than saving a partially completed update.

The background process keeps running independently of the notebook kernel. On macOS it starts `caffeinate` to prevent idle sleep while the process runs. Keep the laptop plugged in and its lid open; restarting or powering off the computer interrupts training. To resume, use `MODE = 'resume'`. Progress is recorded in `training.log`, `status.json` and `history.json` under the run directory. Only the notebook belongs in Git; weights and full checkpoints are local runtime files or release assets.

In [9]:
if MODE == 'review':
    print('Review mode: no training or test evaluation is started.')
elif START_BACKGROUND:
    training_pid = launch_background(MODE)
else:
    run_pipeline(MODE)

Review mode: no training or test evaluation is started.


In [10]:
if (RUN_DIR/'status.json').exists():
    display(Markdown('```json\n'+(RUN_DIR/'status.json').read_text()+'\n```'))
else:
    print('No local run status yet. The published review package can still be used below.')

No local run status yet. The published review package can still be used below.


## 4. Load the selected weights without fine-tuning

The loader verifies the downloaded package's checksums and restores the exact saved architecture and preprocessing. It constructs ResNet18 with `pretrained=False`, loads the trained state dictionary, and calls `eval()`; it does not need to download the original ImageNet weights for inference. If the overnight package is not ready yet, the remaining review plots are skipped with a clear message rather than showing an untrained model as a result.

In [11]:
device = device_for(DEVICE)
resnet_model, resnet_data, resnet_report = load_review_assets()
default_pair = None
if resnet_model is not None:
    resnet_previews = {int(i):im for i,im in zip(resnet_data['preview_indices'],resnet_data['preview_images'])}
    resnet_examples = {int(i):im for i,im in zip(resnet_data['example_indices'],resnet_data['example_images'])}
    default_index = int(resnet_report['gallery_indices']['Resize']['FAKE'])
    default_pair = resnet_pair(default_index,'Resize',1)
    default_stem_channel = int(np.argmax(resnet_data['stem_ablation'][0,:,0]))
    default_last_channel = int(np.argmax(resnet_data['last_ablation'][0,:,0]))
    print(f"Loaded selected epoch {resnet_report['selected_epoch']} on {device}. No fine-tuning was performed.")
    print(resnet_model.net)

No finished model package is available locally yet.


In [12]:
def load_saved_weights(path=WEIGHTS_PATH, selected_device='cpu'):
    """The same save/load pattern as the document-classification notebook."""
    loaded = CifakeResNet18(pretrained=False)
    loaded.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
    return loaded.to(selected_device).eval()

# Example after the run finishes:
# trained_model = load_saved_weights('model_saves/resnet18_plain_seed1.pt')
# with torch.inference_mode():
#     p_real = torch.sigmoid(trained_model(rgb_batch_float_in_0_to_1))

## 5. Learning curves and clean evaluation

The next plots compare training and validation loss and accuracy through the run. The shaded first two epochs indicate classifier warmup, and the dashed line marks the checkpoint selected by validation loss. We will look for the point where validation stops improving while training continues to improve. The table then reports the selected model on validation, the official test set, and the test set with exact repeated images counted once. The test set is evaluated only after checkpoint selection; cached predictions prevent a repeat evaluation during review.

In [13]:
if resnet_model is not None:
    finish_plot(learning_figure(),'learning_curves')
    table(['Split','Images','Accuracy','ROC-AUC (FAKE)','Fake detected','Real false alarms'],
          [(name,resnet_report['evaluation'][key]['n'],f"{resnet_report['evaluation'][key]['accuracy']:.2%}",
            f"{resnet_report['evaluation'][key]['roc_auc_fake']:.4f}",f"{resnet_report['evaluation'][key]['fake_detection_rate']:.2%}",
            f"{resnet_report['evaluation'][key]['real_flagged_fake_rate']:.2%}")
           for name,key in [('Validation','validation'),('Official test','official_test'),('Unique test','unique_test')]])
    print('Reference plain CNN: 94.54% clean validation accuracy; 94.29% official clean test accuracy.')

The confusion matrix below shows the kinds of test mistakes behind the accuracy. Rows are the true labels and columns are predictions. The FAKE→REAL cell counts missed generated images; the REAL→FAKE cell counts false alarms. The threshold remains fixed at 0.5, so the test results are not used to choose a more favourable decision threshold.

In [14]:
if resnet_model is not None:
    finish_plot(confusion_figure(),'test_confusion')

## 6. The same modification tests as the baseline

We will compare both selected models on the exact same 10,000 validation images and 17 conditions: original plus four strengths each of JPEG compression, Gaussian blur, downsize-and-resize, and Gaussian noise. Image identities, noise seeds, JPEG chroma sampling and transformation parameters match the baseline notebook. Edits happen at 32×32; ResNet's fixed enlargement and normalization follow afterwards.

Each column is a modification family. The rows show overall accuracy, the fraction of fake images detected, and the fraction of real images incorrectly flagged. This separates a general improvement from a shift toward calling more images REAL or FAKE. The curves are exploratory validation results, not a newly tuned model or a robustness claim about untested image generators.

In [15]:
if resnet_model is not None:
    finish_plot(robustness_figure(),'validation_robustness')

## 7. Regional explanations and exact output contributions

The pair below is a sensitive fake validation example selected from this ResNet's resize results. We will compare the original with its 32→28→32 version while holding the Grad-CAM target fixed at FAKE, represented by the negative REAL logit. The middle panels show the last residual stage's native **7×7** Grad-CAM map; the right panels enlarge it over the native image. Both versions share a colour scale. Enlargement does not create spatial precision, and positive Grad-CAM intensity does not add up to the probability or prove that an object region caused a decision.

In [16]:
if default_pair is not None:
    finish_plot(gradcam_figure(default_pair),'gradcam_resize')

ResNet18 averages each of its final 512 feature maps and feeds those 512 values to one linear output. We can therefore account for its REAL logit exactly: multiply each pooled activation by its output weight, sum all 512 products, and add the bias. The heatmap below shows every contribution before and after the edit. The bars show the 12 largest changes and the sum of the remaining 500, so together they account for the entire logit change. This is the ResNet equivalent of the baseline's 64-neuron contribution chart; channel numbers do not imply semantic meanings.

In [17]:
if default_pair is not None:
    finish_plot(contribution_figure(default_pair),'output_contributions')

We will also cover native 4×4 regions, moving in steps of two pixels, and measure the FAKE score before covering minus the score after covering. Positive red areas indicate that the covering operation reduces the FAKE score; negative blue areas indicate that it increases it. Overlapping patch effects are averaged. The reference colours are the original image's mean RGB and mid-grey, kept fixed within each before/after pair, and all four maps use one symmetric scale. The mask is applied before enlargement to 224×224. Covering creates artificial edges, so these are responses to a specified intervention rather than pixel-level causal explanations.

In [18]:
if default_pair is not None:
    finish_plot(occlusion_figure(default_pair),'native_occlusion')

## 8. Internal filters and channel interventions

The first grid shows the 64 learned 7×7 stem filters, with adjacent slices for red, green and blue. All use the same signed weight scale. These weights act on normalized, enlarged inputs, and BatchNorm and ReLU follow the convolution, so the weight colours are not class evidence or a complete activation calculation. The selected channel is outlined in green. For the final residual stage, the explorer instead shows each channel's weight into the REAL output; those final channels combine residual paths and are not simply RGB filters.

In [19]:
if default_pair is not None:
    finish_plot(weight_figure('stem',default_stem_channel),'stem_weights')

The next view follows the selected stem channel on the same original/modified pair. Its native response maps use a shared scale, and the input labels show the prediction with the full channel set to zero. The stem response is measured after Conv–BatchNorm–ReLU; the explorer also offers the final layer4 output after residual additions and ReLU. A bright activation is not automatically important for the final decision, which is why we place the intervention's prediction beside the map.

In [20]:
if default_pair is not None:
    finish_plot(channel_figure(default_pair,'stem',default_stem_channel),'stem_response')

To investigate the pattern a channel responds to, we will show its three highest spatial activations among the same seeded, balanced 2,000-image validation sample used for the baseline's internal investigation. Selection is based on response strength, not label or correct classification. The image and response map remain separate because the final residual features have broad receptive fields; we do not draw precise object boundaries from them. Several examples can suggest a pattern to test, but do not justify assigning a semantic class label to a channel.

In [21]:
if default_pair is not None:
    finish_plot(examples_figure('stem',default_stem_channel),'strongest_examples')

The aggregate intervention below uses those same 2,000 validation images in original, resized and JPEG-quality-60 conditions. Every one of the 64 stem channels and 512 final channels is zeroed separately. Stem ablation runs the remaining residual network again. Final-channel ablation can be calculated exactly by subtracting that channel's pooled-activation × output-weight contribution from the logit, because only average pooling and the linear output remain.

Positive accuracy loss means removal hurts; negative values mean removal helps in that condition. For readability, the static figure shows the 20 largest absolute clean-accuracy effects in each stage, preserving their identities across conditions. The explorer exposes every channel and reports new mistakes, repaired mistakes and class-specific rates. These are descriptive, paired sample effects; channels can compensate for one another, and removal is an artificial internal state, not evidence that pruning and retraining will improve the model.

In [22]:
if default_pair is not None:
    finish_plot(internal_figure(),'channel_ablation')

## 9. Interactive comparison

Choose a preview image, modification and strength, plus a stem or final feature channel, then press **Inspect**. The tabs connect Grad-CAM, exact output contributions, native occlusion, channel responses, weights and strongest examples. Strength zero compares an image with itself. Patch sizes 4 and 8 use strides 2 and 4. Changing controls marks the previous result as stale until the update finishes. The aggregate ablation tabs always use the fixed three conditions and the 2,000-image sample; they do not represent only the selected preview image. The controls require a live kernel, while the saved static figures can be read without one.

In [23]:
if default_pair is not None:
    resnet_explorer = draw_explorer()

## 10. Team sharing and scope of conclusions

The normal completion step exports an executed review notebook, a plain `.pt` state dictionary, the saved analysis and a checksum manifest. The [team release](https://github.com/adamp-22/kandy/releases/tag/resnet18-plain-seed1-v1) contains these assets individually and in `resnet18_team_bundle.zip`. Unzip the bundle and run the notebook in review mode; its relative directory structure makes the local assets available without downloading or fine-tuning anything. The simple weights are also under `model_saves/`, following the document-classification notebook's convention. Full optimizer checkpoints stay in `runs/` for recovery and can be copied separately if a teammate needs to resume training.

The clean test evaluates the checkpoint chosen using validation loss. The modification curves and channel investigations use validation data and remain exploratory. Any advantage over the plain CNN includes ImageNet pretraining and different preprocessing as well as a larger residual architecture. The saved diagnostics are intended to show where behaviour differs and motivate controlled follow-up experiments, not to establish detection of arbitrary future image generators.